<a href="https://colab.research.google.com/github/Knguyen351/Advanced-Programming-311-Streamlit-Project/blob/main/Notebook_2_Prototype_311_Dashboard_KevinNguyen11-02-26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 2: Prototype Your 311 Dashboard

## Purpose
Before you build a Streamlit application, make sure the **date filtering and charts work in Python**.

This notebook intentionally stays simple. You will:
1. Read the reduced file created in Notebook 1.
2. Choose a start date and end date within 2025.
3. Filter the data to those dates.
4. Display **two simple charts**.
5. Replace the sample charts with two charts you want to use in your own dashboard.

Later, ChatGPT can help you turn this working logic into `app.py`.


## Project overview and intended audience

**311 Service Request Patterns (October–December 2025)** examines three questions:

1. Which 311 service types are most frequent on each weekday?
2. Which service types lead on a selected weekday, especially after excluding Information Request?
3. How does request volume change over time (daily versus weekly)?

**Intended audience:** municipal service planners, public-sector data analysts, and residents interested in service demand. Counts represent records of requests, **not necessarily telephone calls**. This is a reduced dataset; conclusions apply to this subset and selected dates.


## 1. Import the libraries and read your reduced dataset


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Upload your reduced CSV beside this notebook before running.
DATA_FILE = Path("dashboard_data.csv")
if not DATA_FILE.exists():
    matches = list(Path(".").glob("311_2025_dashboard*.csv"))
    if not matches:
        raise FileNotFoundError("Upload dashboard_data.csv or the original 311 CSV first.")
    DATA_FILE = matches[0]

df = pd.read_csv(DATA_FILE)
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
df.head()


## 2. Convert `request_date` back to a datetime
CSV files do not preserve pandas data types, so we convert this field after reading the file.


In [ ]:
df["request_date"] = pd.to_datetime(df["request_date"], errors="coerce")

print("Available dates:")
print(df["request_date"].min(), "through", df["request_date"].max())


## 3. Select a start date and end date
For now, you will change two Python variables. **Change these dates and rerun the cells below.**

When you build the Streamlit application, these variables will become interactive date controls.


In [ ]:
# CHANGE THESE DATES to explore a different part of 2025
START_DATE = "2025-10-01"
END_DATE = "2025-12-31"

start_date = pd.to_datetime(START_DATE)
end_date = pd.to_datetime(END_DATE)

if start_date > end_date:
    raise ValueError("START_DATE must be on or before END_DATE.")


## 4. Filter the DataFrame
This is the core idea that your Streamlit dashboard will use.


In [ ]:
# Use START_DATE and END_DATE defined above; do not overwrite them here.
filtered_df = df[
    (df["request_date"] >= start_date) &
    (df["request_date"] < end_date + pd.Timedelta(days=1))
].copy()

print(f"Selected period: {START_DATE} through {END_DATE}")
print(f"Requests in selected period: {len(filtered_df):,}")


## 5. Three interactive charts
These three charts examine service categories by weekday, the most requested services, and trends over time. Change the dates above and rerun the cells below.


## Summary metrics
These metrics update when you change the notebook date range and rerun from the filter cell.


In [ ]:
# Three useful summary metrics for the selected dates
if filtered_df.empty:
    print("No records in the selected date range. Change START_DATE and END_DATE.")
else:
    counts = filtered_df["service_name"].fillna("Unknown").value_counts()
    weekdays = filtered_df["request_date"].dt.day_name().value_counts()
    print(f"Total requests: {len(filtered_df):,}")
    print(f"Busiest weekday: {weekdays.idxmax()} ({weekdays.max():,} requests)")
    print(f"Most requested service: {counts.idxmax()} ({counts.max():,} requests)")


In [ ]:

# YOUR CHART 1 - Interactive Heatmap with Filters

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import interact, DatePicker, Checkbox

days = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

def plot_heatmap(start, end, exclude_info):

    if start is None or end is None:
        print("Please select both dates.")
        return

    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    if start > end:
        print("Start date cannot be after end date.")
        return

    # Filter requests by selected dates
    data = filtered_df[
        filtered_df["request_date"].between(start, end)
    ].copy()

    # Exclude Information Request if selected
    if exclude_info:
        data = data[
            data["service_name"].str.strip().str.casefold()
            != "information request"
        ]

    if data.empty:
        print("No requests available for these filters.")
        return

    # Find top 10 after filtering
    top_10 = (
        data["service_name"]
        .value_counts()
        .head(10)
        .index
    )

    # Create heatmap data
    heatmap_data = pd.crosstab(
        data["service_name"],
        data["day_of_week"]
    )

    # Arrange services and weekdays
    heatmap_data = heatmap_data.reindex(
        index=top_10,
        columns=days,
        fill_value=0
    )

    print(f"Selected Period: {start.date()} to {end.date()}")
    print(f"Total Requests: {len(data):,}")
    print(f"Information Requests Excluded: {exclude_info}")

    # Display heatmap
    plt.figure(figsize=(12, 7))

    sns.heatmap(
        heatmap_data,
        annot=True,
        fmt="d",
        cmap="Blues",
        linewidths=0.5
    )

    plt.title("Top 10 Types of 311 Requests by Day of Week")
    plt.xlabel("Day of Week")
    plt.ylabel("Service Type")
    plt.xticks(rotation=30)

    plt.tight_layout()
    plt.show()


# Interactive controls
interact(
    plot_heatmap,

    start=DatePicker(
        value=filtered_df["request_date"].min().date(),
        description="Start Date:"
    ),

    end=DatePicker(
        value=filtered_df["request_date"].max().date(),
        description="End Date:"
    ),

    exclude_info=Checkbox(
        value=False,
        description="Exclude Information Request"
    )
)


In [ ]:

# YOUR CHART 2 - Interactive Service Requests

import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown, Checkbox

def explore_requests(day="All Days", top_n=10, exclude_info=False):

    data = filtered_df.copy()

    # Exclude Information Request if selected
    if exclude_info:
        data = data[
            data["service_name"].str.strip().str.casefold()
            != "information request"
        ]

    # Filter by weekday
    if day != "All Days":
        data = data[data["day_of_week"] == day]

    if data.empty:
        print("No requests available for this selection.")
        return

    # Count service requests
    service_counts = data["service_name"].value_counts()

    print("Selected Day:", day)
    print(f"Total Requests: {len(data):,}")
    print("Most Requested Service:", service_counts.idxmax())
    print(f"Number of Requests: {service_counts.max():,}")

    # Display top service requests
    top_services = service_counts.head(top_n).sort_values()

    fig, ax = plt.subplots(figsize=(10, 6))

    bars = ax.barh(
        top_services.index,
        top_services.values,
        color="skyblue",
        edgecolor="black"
    )

    ax.bar_label(bars, padding=3)

    ax.set_title(f"Top {top_n} Service Requests - {day}")
    ax.set_xlabel("Number of Requests")
    ax.set_ylabel("Service Type")
    ax.set_xlim(0, top_services.max() * 1.2)

    plt.tight_layout()
    plt.show()


interact(
    explore_requests,

    day=Dropdown(
        options=["All Days"] + days,
        description="Select Day:"
    ),

    top_n=Dropdown(
        options=[5, 10, 15],
        value=10,
        description="Top Requests:"
    ),

    exclude_info=Checkbox(
        value=False,
        description="Exclude Information Request"
    )
)


In [ ]:

# YOUR CHART 3 - Interactive 311 Request Trends

import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, DatePicker, Dropdown, Checkbox

# Available service types
services = ["All Services"] + sorted(
    filtered_df["service_name"].dropna().unique().tolist()
)

def plot_trends(start, end, service, view_by, exclude_info):

    if start is None or end is None:
        print("Please select both dates.")
        return

    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    if start > end:
        print("Start date cannot be after end date.")
        return

    # Filter by dates
    data = filtered_df[
        filtered_df["request_date"].between(start, end)
    ].copy()

    # Exclude Information Request
    if exclude_info:
        data = data[
            data["service_name"].str.strip().str.casefold()
            != "information request"
        ]

    # Filter by service type
    if service != "All Services":
        data = data[data["service_name"] == service]

    if data.empty:
        print("No data available for this selection.")
        return

    # Count requests per date
    daily_dates = pd.date_range(start, end, freq="D")

    daily_counts = (
        data.groupby("request_date")
        .size()
        .reindex(daily_dates, fill_value=0)
    )

    # Daily or weekly
    if view_by == "Weekly":
        trend = daily_counts.resample("W-SUN").sum()
    else:
        trend = daily_counts

    # Summary
    print(f"Selected Service: {service}")
    print(f"Total Requests: {len(data):,}")
    print(f"Average Requests per {view_by[:-2] if view_by == 'Weekly' else 'Day'}: {trend.mean():,.1f}")

    # Line chart
    fig, ax = plt.subplots(figsize=(12, 5))

    ax.plot(
        trend.index,
        trend.values,
        marker="o",
        color="royalblue",
        linewidth=2
    )

    ax.set_title(f"311 Request Trends - {service}")
    ax.set_xlabel("Date")
    ax.set_ylabel("Number of Requests")
    ax.grid(True, alpha=0.3)

    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()


interact(
    plot_trends,

    start=DatePicker(
        value=filtered_df["request_date"].min().date(),
        description="Start Date:"
    ),

    end=DatePicker(
        value=filtered_df["request_date"].max().date(),
        description="End Date:"
    ),

    service=Dropdown(
        options=services,
        description="Service Type:"
    ),

    view_by=Dropdown(
        options=["Daily", "Weekly"],
        value="Weekly",
        description="View By:"
    ),

    exclude_info=Checkbox(
        value=False,
        description="Exclude Information Request"
    )
)


## Findings and interpretation
These statements are computed from the active selection, so they do not become stale when you change the date range. The heatmap highlights weekday/service combinations; the bar chart ranks service types; and the trend chart shows time variation.


In [ ]:
# Data-driven findings for the selected period
if filtered_df.empty:
    print("No observations for the selected date range.")
else:
    weekday_counts = filtered_df["request_date"].dt.day_name().value_counts()
    service_counts = filtered_df["service_name"].value_counts()
    without_info = filtered_df.loc[
        filtered_df["service_name"].fillna("").str.strip().str.casefold() != "information request"
    ]
    print(f"Weekday pattern: {weekday_counts.idxmax()} has the most requests ({weekday_counts.max():,}) and {weekday_counts.idxmin()} the fewest ({weekday_counts.min():,}).")
    print(f"Leading service category: {service_counts.idxmax()} ({service_counts.max():,} records).")
    if not without_info.empty:
        other = without_info["service_name"].value_counts()
        print(f"Without Information Request: {other.idxmax()} is the leading category ({other.max():,}).")
    monthly = filtered_df.groupby(filtered_df["request_date"].dt.to_period("M")).size()
    if len(monthly):
        print(f"Trend context: {monthly.idxmax()} is the highest-volume selected month ({monthly.max():,} requests).")
    print("Interpretation caution: weekday totals are cumulative across the period, not averages per weekday occurrence.")


## 8. Test Your Date Filter
Before moving on:
1. Change `START_DATE` and `END_DATE` near the top of the notebook.
2. Rerun the filter and chart cells.
3. Confirm that the number of records and charts change.

If that works, you have the core logic for your Streamlit dashboard.


## Streamlit and GitHub submission
Use the accompanying `app.py`, `dashboard_data.csv`, `requirements.txt`, and `README.md`. Deploy to Streamlit Community Cloud and add the live URL to the README. Test every interactive control in the published app.


## AI use statement
I used ChatGPT to assist with Python/Streamlit code, debugging, dashboard layout, and documentation. I am responsible for verifying the results, understanding the charts, and testing the published app.
